In [ ]:
from notebooks_function import *

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pypsa
import warnings
import country_converter as coco
import os
import matplotlib.dates as mdates

cc = coco.CountryConverter()


def get_time_series(n, df, df_load, start_time, end_time, title):
    
    df_pos = df.clip(lower=0)
    df_neg = df.clip(upper=0)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    df_pos.loc[:,start_time:end_time].T.plot(
        kind="area",
        ax=ax,
        color=[n.carriers.color[c] for c in df_pos.index]
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", 
            message="Attempting to set identical low and high ylims"
        )
        
        df_neg.loc[:,start_time:end_time].T.plot(
            kind="area",
            ax=ax,
            color=[n.carriers.color[c] for c in df_neg.index],
            legend=False
        )
    
    # Plot load demand (overlaid)
    df_load.loc[start_time:end_time].plot(
        ax=ax,
        color="black",
        linewidth=2,
        style='--',
        alpha=0.8,
    )
    
    # ---- Legend Formatting ----
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = handles[::-1], labels[::-1]  # Reverse for nicer order
    
    # Remove duplicates while preserving order (reversed)
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if l not in seen and not seen.add(l)]
    
    # Unzip the filtered handles and labels
    handles, labels = zip(*unique)
    
    # Replace technical carrier names with nice names if available
    labels = [
        n.carriers.nice_name.get(label, label) #.title()
        for label in labels
    ]
    
    ax.legend(
        handles,
        labels,
        title="Carrier",
        bbox_to_anchor=(1.0, 0.85),
        loc='upper left'
    )
    
    ymax = df_pos.sum().max()
    ymin = df_neg.sum().min()
    
    delta = abs(ymax) * 0.1
    ymin -= delta
    ymax += delta

    time = df_load.loc[start_time:end_time].index
    xmin = time[0]
    xmax = time[-1]
    
    ax.set_ylim(ymin, ymax)
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.margins(x=0)
    ax.set_ylabel("Generation [GW]")
    ax.grid()

    ax.set_title(title)
    
    fig.tight_layout()

    return fig


# 1 - Retrieve networks

In [ ]:
# For all scenarios with 4 timesteps
years = [2025, 2030, 2035, 2040]
scenarios = [
    "baseline",
    "baseline-rps",
    "baseline-co2-price25",
    "baseline-co2-price50",
    "baseline-co2-price100",
    "energy-match-25",
    "hourly-match-25-90",
    "hourly-match-25-95",
    "hourly-match-25-98",
    "hourly-match-25-99",
    "hourly-match-EU-25-99",
    "hourly-match-no-LDES-25-99",
    "hourly-match-no-clean-firm-25-99",
    "hourly-match-co2-price25-25-99",
    "hourly-match-co2-price50-25-99",
    "hourly-match-co2-price100-25-99",
    "hourly-match-noadd-10-99",
    "hourly-match-noadd-50-99",
    "hourly-match-noadd-90-99",
]

# For all scenarios with 2 timesteps
# years = [2025, 2030]
# scenarios = ["energy-match-50",
#              "hourly-match-50-90",
#              "hourly-match-50-95",
#              "hourly-match-50-98",
#              "hourly-match-50-99",
#              "hourly-match-EU-50-99",
#              "hourly-match-no-LDES-50-99",
#              "hourly-match-no-clean-firm-50-99",
#              "hourly-match-co2-price25-50-99",
#              "hourly-match-co2-price50-50-99",
#              ]

# Build MultiIndex
index = pd.MultiIndex.from_product(
    [years, scenarios],
    names=["year", "scenario"]
)

# Create empty DataFrame
df_networks = pd.DataFrame(index=index, columns=["network"])

# Fill it
for year, sc in index:
    try:
        n = pypsa.Network(f"../results/{sc}/networks/base_s_39___{year}.nc")
    except:
        print(f"{sc}-{year} not availabe")
        continue
    n = prepare_network(n)
    n = drop_year(n)
    n.name = f"{sc}-{year}"
    df_networks.loc[(year, sc), "network"] = n

    m = strip_network_GoO(n)
    m.name = "GoO-" + m.name
    df_networks.loc[(year, sc), "GoO"] = m

df_networks = df_networks.dropna()

In [ ]:
date_time = ('05-01','05-14')
start_time = pd.to_datetime(date_time[0], format='%m-%d').strftime("%m-%d %H:%M")
end_time = pd.to_datetime(date_time[1], format='%m-%d').strftime("%m-%d %H:%M")

save_fig = True
show_fig = False

## GO Market Hourly Matching

In [ ]:
# Precompute GoO stats
df_GoO, _ = get_stats_all(
    df_networks["GoO"], 
    "energy_balance", 
    groupby=["country", "carrier"], 
    aggregate_time=False
)

# Get sorted list of countries, excluding empty strings
countries = ["EU"] + sorted(set(df_networks.iloc[0, 0].buses.country.unique()) - {""})

fig_base_path = "figures/GoO_time_series/country"

for year, scenario in df_networks.index:
    if "baseline" in scenario:
        continue

    # Filter and clean data
    df_balance_all = clean_virtual_names(df_GoO[(year, scenario)])

    # Load GO demand
    m = df_networks.loc[(year, scenario), "GoO"]

    for country in countries:
        
        if country == "EU":
            df_country = df_balance_all.copy(deep=True)
            df_load = m.loads_t.p_set.filter(like="GO Demand").T.sum() * 1e-3
            cc_name = country
        else: 
            df_country = df_balance_all.xs(country, level="country")
            df_load = m.loads_t.p_set[f"GO Demand {country} New"] * 1e-3
            cc_name = cc.convert(country, to="short_name")

        df_country = df_country.groupby("carrier").sum()
        df_country = df_country[df_country.index != "GoO"] * 1e-3

        # calculate hourly matching
        result = pd.concat([df_load, df_country.sum()], axis=1).min(axis=1)
        weighting = n.snapshot_weightings.objective 
        hourly_matching = round((weighting @ result) / (weighting @ df_load) * 100, 2)
    
        # Generate and save figure
        fig = get_time_series(
            m, 
            df_country,
            df_load,
            start_time,
            end_time,
            title=f"GO Time Series - {cc_name}: {hourly_matching} Hourly Matching"
        )

        title_fig = f"GO Time Series - {scenario} - {year} - {cc_name}"

        if save_fig:
            country_fig_path = os.path.join(fig_base_path, cc_name, scenario)
            os.makedirs(country_fig_path, exist_ok=True)
            fig.savefig(os.path.join(country_fig_path, f"{title_fig}.png"), dpi=300, bbox_inches="tight")

        if not show_fig:
            plt.close()


## Total electricity market

In [ ]:
elec_demand = [
    "electricity",
    "land transport EV",
    "industry electricity",
    "agriculture electricity",
    "agriculture machinery electric",
]

emitters = ["CCGT", "OCGT", "coal", "lignite", "oil"]
                  
# Get statistics
df, _ = get_stats_all(
    df_networks["network"], 
    "energy_balance", 
    groupby=["country","bus_carrier","carrier"], 
    aggregate_time=False
)

df_load_all, _ = get_stats_all(
    df_networks["network"], 
    "energy_balance", 
    groupby=["country"], 
    components=["Load"], 
    carrier=elec_demand, 
    aggregate_time=False
)

# Get sorted list of countries excluding empty strings
countries = ["EU"] + sorted(set(df_networks.iloc[0, 0].buses.country.unique()) - {""})

fig_base_path = "figures/time_series/country"

for year, scenario in df_networks.index:

    title_fig = f"Time Series - {scenario} - {year}"

    # Filter df for country and relevant bus_carrier/carrier
    df_balance_all = df[(year, scenario)]

    for country in countries:
        
        if country == "EU":
            df_balance = df_balance_all.copy(deep=True)
            df_balance.index = df_balance.index.droplevel("country")
            df_balance = df_balance.groupby(level=df_balance.index.names).sum()
            df_load = -df_load_all[(year, scenario)].groupby("country").sum().sum()
            cc_name = country

        else:
            df_balance = df_balance_all.xs(country, level="country")
            df_load = -df_load_all[(year, scenario)].groupby("country").sum().loc[country]
            cc_name = cc.convert(country, to="short_name")

        # print(f"-------------------------------------Analyzing {cc_name}-------------------------------------")
            
        df_balance = df_balance[
            df_balance.index.get_level_values("bus_carrier").isin(["AC", "low voltage"])
            & ~df_balance.index.get_level_values("carrier").isin([
                "AC", "DC", "electricity", "low voltage", 
                "electricity distribution grid", "BEV charger",
                "home battery charger", "home battery discharger"
            ] + emitters)
        ]
    
        # Aggregate by carrier
        df_country = df_balance.groupby("carrier").sum()
        df_country = df_country.rename(index=grouping_storage).groupby("carrier").sum()

        # calculate hourly matching
        result = pd.concat([df_load, df_country.sum()], axis=1).min(axis=1)
        weighting = m.snapshot_weightings.objective 
        hourly_matching = round((weighting @ result) / (weighting @ df_load) * 100, 2)
        
        # Generate figure
        fig = get_time_series(
            m, 
            df_country,
            df_load,
            start_time,
            end_time,
            title=f"Time Series - {cc_name}: {hourly_matching} Max Energy Matching"
        )
    
        if save_fig:
            country_fig_path = os.path.join(fig_base_path, cc_name, scenario)
            os.makedirs(country_fig_path, exist_ok=True)
            fig.savefig(os.path.join(country_fig_path, f"{title_fig}.png"), dpi=300, bbox_inches="tight")

        if not show_fig:
            plt.close()